In [10]:
import argparse
import re
import shutil
from functools import partial
from pathlib import Path

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from lerobot.constants import HF_LEROBOT_HOME
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from oxe_utils.configs import OXE_DATASET_CONFIGS, ActionEncoding, StateEncoding
from oxe_utils.transforms import OXE_STANDARDIZATION_TRANSFORMS
from lerobot.datasets.utils import append_jsonlines
np.set_printoptions(precision=2)


import json
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import mediapy
from scipy.spatial.transform import Rotation as R
import cv2
import imageio

In [11]:
args = {
    "raw_dir": Path("/mnt/hwfile/tangyuhang/droid/1.0.0"),
    "local_dir": Path("/mnt/hwfile/tangyuhang/droid/droid_lerobot"),
    "repo_id": "luanqibazao",
    "use_videos": True,
    
}

default_args = {
    "robot_type": None,
    "fps": None,
    "image_writer_process": 5,
    "image_writer_threads": 10,
    
}

raw_dir = args["raw_dir"]
local_dir = args["local_dir"]
repo_id = args["repo_id"]
use_videos = args["use_videos"]

robot_type = default_args["robot_type"]
fps = default_args["fps"]
image_writer_process = default_args["image_writer_process"]
image_writer_threads = default_args["image_writer_threads"]

In [12]:
last_part = raw_dir.name
if re.match(r"^\d+\.\d+\.\d+$", last_part):
    version = last_part
    dataset_name = raw_dir.parent.name
    data_dir = raw_dir.parent.parent
else:
    version = ""
    dataset_name = last_part
    data_dir = raw_dir.parent

if local_dir is None:
    local_dir = Path(HF_LEROBOT_HOME)
local_dir /= f"{dataset_name}_{version}_lerobot"
if local_dir.exists():
    shutil.rmtree(local_dir)

In [13]:
print(f"version: {version}\n")
print(f"dataset_name: {dataset_name}\n")
print(f"data_dir: {data_dir}\n")
print(f"local_dir: {local_dir}\n")

version: 1.0.0

dataset_name: droid

data_dir: /mnt/hwfile/tangyuhang

local_dir: /mnt/hwfile/tangyuhang/droid/droid_lerobot/droid_1.0.0_lerobot



In [14]:
def transform_raw_dataset(episode, dataset_name):
    traj = next(iter(episode["steps"].batch(episode["steps"].cardinality())))

    if dataset_name in OXE_STANDARDIZATION_TRANSFORMS:
        traj = OXE_STANDARDIZATION_TRANSFORMS[dataset_name](traj)

    if dataset_name in OXE_DATASET_CONFIGS:
        state_obs_keys = OXE_DATASET_CONFIGS[dataset_name]["state_obs_keys"]
    else:
        state_obs_keys = [None for _ in range(8)]

    proprio = tf.concat(
        [
            (
                tf.zeros((tf.shape(traj["action"])[0], 1), dtype=tf.float32)  # padding
                if key is None
                else tf.cast(traj["observation"][key], tf.float32)
            )
            for key in state_obs_keys
        ],
        axis=1,
    )

    traj.update(
        {
            "proprio": proprio,
            "task": traj.pop("language_instruction"),
            "action": tf.cast(traj["action"], tf.float32),
        }
    )

    episode["steps"] = traj
    return episode


def generate_features_from_raw(builder: tfds.core.DatasetBuilder, use_videos: bool = True):
    dataset_name = Path(builder.data_dir).parent.name

    state_names = [f"motor_{i}" for i in range(8)]
    if dataset_name in OXE_DATASET_CONFIGS:
        state_encoding = OXE_DATASET_CONFIGS[dataset_name]["state_encoding"]
        if state_encoding == StateEncoding.POS_EULER:
            state_names = ["x", "y", "z", "roll", "pitch", "yaw", "pad", "gripper"]
            if "libero" in dataset_name:
                state_names = ["x", "y", "z", "roll", "pitch", "yaw", "gripper", "gripper"]  # 2D gripper state
        elif state_encoding == StateEncoding.POS_QUAT:
            state_names = ["x", "y", "z", "rx", "ry", "rz", "rw", "gripper"]
        elif state_encoding == StateEncoding.JOINT:
            state_names = [f"motor_{i}" for i in range(7)] + ["gripper"]
            state_obs_keys = OXE_DATASET_CONFIGS[dataset_name]["state_obs_keys"]
            pad_count = state_obs_keys[:-1].count(None)
            state_names[-pad_count - 1 : -1] = ["pad"] * pad_count
            state_names[-1] = "pad" if state_obs_keys[-1] is None else state_names[-1]

    action_names = [f"motor_{i}" for i in range(8)]
    if dataset_name in OXE_DATASET_CONFIGS:
        action_encoding = OXE_DATASET_CONFIGS[dataset_name]["action_encoding"]
        if action_encoding == ActionEncoding.EEF_POS:
            action_names = ["x", "y", "z", "roll", "pitch", "yaw", "gripper"]
        elif action_encoding == ActionEncoding.JOINT_POS:
            action_names = [f"motor_{i}" for i in range(7)] + ["gripper"]

    DEFAULT_FEATURES = {
        "observation.state": {
            "dtype": "float32",
            "shape": (len(state_names),),
            "names": {"motors": state_names},
        },
        "action": {
            "dtype": "float32",
            "shape": (len(action_names),),
            "names": {"motors": action_names},
        },
    }

    obs = builder.info.features["steps"]["observation"]
    features = {
        f"observation.images.{key}": {
            "dtype": "video" if use_videos else "image",
            "shape": value.shape,
            "names": ["height", "width", "rgb"],
        }
        for key, value in obs.items()
        if "depth" not in key and any(x in key for x in ["image", "rgb"])
    }
    return {**features, **DEFAULT_FEATURES}

In [15]:
builder = tfds.builder(dataset_name, data_dir=data_dir, version=version)
features = generate_features_from_raw(builder, use_videos)
filter_fn = lambda e: e["success"] if dataset_name == "kuka" else True
raw_dataset = (
    builder.as_dataset(split="train")
    .filter(filter_fn)
    .map(partial(transform_raw_dataset, dataset_name=dataset_name))
)

I0000 00:00:1767786801.652130   24099 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 77945 MB memory:  -> device: 0, name: NVIDIA A800-SXM4-80GB, pci bus id: 0000:8f:00.0, compute capability: 8.0


In [17]:
for example in raw_dataset.take(1):
    print("Keys:", example.keys())
    print("Episode metadata:")
    for k, v in example['episode_metadata'].items():
        print(f"  {k}: {v.numpy().decode('utf-8')}")

    print("Steps")
    for k, v in example['steps'].items():
        print(f"  {k} ")
        
    print("action_dict of Steps")
    for k, v in example['steps']['action_dict'].items():
        print(f"  {k} ")

    print(f"\naction: {example['steps']['action'][:3]}\n")
    print(f"\nobservation: {example['steps']['proprio'][:3]}")
    # print(f"\ntask: {example['steps']['task'][0].numpy().decode()}")
    print(example['steps']['observation']['exterior_image_1_left'])
    

Keys: dict_keys(['episode_metadata', 'steps'])
Episode metadata:
  file_path: gs://xembodiment_data/r2d2/r2d2-data-full/TRI/success/2024-02-08/Thu_Feb__8_16:38:30_2024/trajectory.h5
  recording_folderpath: gs://xembodiment_data/r2d2/r2d2-data-full/TRI/success/2024-02-08/Thu_Feb__8_16:38:30_2024/recordings/MP4
Steps
  action 
  action_dict 
  discount 
  is_first 
  is_last 
  is_terminal 
  language_instruction_2 
  language_instruction_3 
  observation 
  reward 
  proprio 
  task 
action_dict of Steps
  cartesian_position 
  cartesian_velocity 
  gripper_position 
  gripper_velocity 
  joint_position 
  joint_velocity 

action: [[0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1.]]


observation: [[ 0.39 -0.01  0.54  3.13 -0.05  0.06  0.    0.  ]
 [ 0.39 -0.01  0.54  3.13 -0.05  0.06  0.    0.  ]
 [ 0.39 -0.01  0.54  3.13 -0.05  0.06  0.    0.  ]]
tf.Tensor(
[[[[101  98  81]
   [ 94  91  74]
   [119 113  97]
   ...
   [ 95  96  90]
   [ 95  96  90]
   [ 96  97  91]]

2026-01-07 11:53:50.267791: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [18]:
if fps is None:
    if dataset_name in OXE_DATASET_CONFIGS:
        fps = OXE_DATASET_CONFIGS[dataset_name]["control_frequency"]
    else:
        fps = 10

if robot_type is None:
    if dataset_name in OXE_DATASET_CONFIGS:
        robot_type = OXE_DATASET_CONFIGS[dataset_name]["robot_type"]
        robot_type = robot_type.lower().replace(" ", "_").replace("-", "_")
    else:
        robot_type = "unknown"

# 

In [19]:
print(f"robot_type: {robot_type}")
print(f"fps:        {fps}")

robot_type: franka
fps:        15


In [10]:
lerobot_dataset = LeRobotDataset.create(
    repo_id=repo_id,
    robot_type=robot_type,
    root=local_dir,
    fps=int(fps),
    use_videos=use_videos,
    features=features,
    image_writer_threads=image_writer_threads,
    image_writer_processes=image_writer_process,
)

Process Process-3:
Process Process-2:
Process Process-4:
Process Process-1:
Process Process-5:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/mnt/petrelfs/tangyuhang/miniconda3/envs/lerobot/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/mnt/petrelfs/tangyuhang/miniconda3/envs/lerobot/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/petrelfs/tangyuhang/miniconda3/envs/lerobot/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/mnt/petrelfs/tangyuhang/miniconda3/envs/lerobot/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/mnt/petrelfs/tangyuhang/miniconda3/envs/lerobot/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File

In [16]:
for episode in raw_dataset.as_numpy_iterator():
    traj = episode["steps"]
    for i in range(traj["action"].shape[0]):
        image_dict = {
            f"observation.images.{key}": value[i]
            for key, value in traj["observation"].items()
            if "depth" not in key and any(x in key for x in ["image", "rgb"])
        }
        lerobot_dataset.add_frame(
            {
                **image_dict,
                "observation.state": traj["proprio"][i],
                "action": traj["action"][i],
            },
            task=traj["task"][0].decode(),
        )
    lerobot_dataset.save_episode()

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 12.87ba/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[info]: Level of Parallelism: 6
Svt[info]: Number of PPCS 305
Svt[info]: [asm level on system : up to avx512icl]
Svt[info]: [asm level selected : up to avx512icl]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 320 / 184 / 15 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 8 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-frame type 			: 2 / 32 / key frame
Svt[info]

KeyboardInterrupt: 

In [15]:
! pip list

Package                      Version     Editable project location
---------------------------- ----------- ----------------------------------------
absl-py                      2.3.1
accelerate                   1.11.0
aiohappyeyeballs             2.6.1
aiohttp                      3.13.1
aiosignal                    1.4.0
annotated-types              0.7.0
anyio                        4.12.0
argon2-cffi                  25.1.0
argon2-cffi-bindings         25.1.0
array_record                 0.8.1
arrow                        1.4.0
asttokens                    3.0.1
astunparse                   1.6.3
async-lru                    2.0.5
async-timeout                5.0.1
attrs                        25.4.0
av                           15.1.0
babel                        2.17.0
beautifulsoup4               4.14.3
bleach                       6.3.0
blinker                      1.9.0
certifi                      2025.10.5
cffi                         2.0.0
charset-normalizer           3.4.